# POC for distributed data alignment


## Preparation of the dataset

We use Adult as typical benchmark dataset.

### Dataset loading

We are only interested in the features

In [1]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo("adult")

X = adult.data.features

X

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States


### Parties creation

Let's split the dataset and create different parties relying on the same schema (to begin with)

In [2]:
import numpy as np

A, B, C = np.array_split(X, 3)

/Users/stefano/Projects/PrivFusion/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


And let us augment the dataset adding a date column, for no particular reason other than it is useful

In [3]:
import datetime
import random


def generate_dates(many: int, format: str = r"%Y-%m-%d") -> list[str]:
    return [
        datetime.date(
            int(1980 + 50 * random.random()), int(6 + random.random() * 6), int(15 + 10 * random.random())
        ).strftime(format)
        for _ in range(many)
    ]

In [4]:
A["date"] = generate_dates(len(A.index), r"%A %d %B %y")

A["date"]

0          Saturday 23 October 27
1             Sunday 19 August 07
2            Tuesday 16 August 11
3         Thursday 22 November 18
4        Saturday 18 September 27
                   ...           
16276          Tuesday 16 June 26
16277        Friday 21 October 11
16278     Saturday 19 November 16
16279       Monday 15 November 93
16280       Friday 18 November 22
Name: date, Length: 16281, dtype: object

In [5]:
B["date"] = generate_dates(len(B.index), r"%d-%b-%Y")

B["date"]

16281    18-Oct-2023
16282    21-Jul-1999
16283    20-Sep-2024
16284    15-Jul-1993
16285    20-Jun-1982
            ...     
32557    24-Oct-1988
32558    21-Aug-1986
32559    21-Nov-2022
32560    22-Oct-2019
32561    22-Nov-1998
Name: date, Length: 16281, dtype: object

In [6]:
C["date"] = generate_dates(len(C.index), r"%Y-%M-%d")

C["date"]

32562    2006-00-23
32563    2026-00-24
32564    1984-00-18
32565    2025-00-20
32566    2021-00-19
            ...    
48837    2002-00-17
48838    1995-00-17
48839    2012-00-22
48840    2010-00-16
48841    1999-00-24
Name: date, Length: 16280, dtype: object

In [8]:
from privfusion.utils import extract_types

extract_types(B)

['Numeric',
 None,
 'Numeric',
 None,
 'Numeric',
 None,
 None,
 None,
 'Etnicity',
 'Gender',
 'Numeric',
 'Numeric',
 'Numeric',
 'Country',
 'DateTime']